# M3L3 E10 - Support bot baseline (Resolution)
### Módulo 3 - Lecture 3 - Sistemas Multiagente

## Qué vas a aprender hoy

- Medir un bot único antes de refactorizar a multiagente.
- Entender por qué necesitamos un baseline.
- Crear un benchmark fijo de consultas.
- Calcular accuracy simple.
- Detectar el límite de un agente único cuando aparece una consulta mixta.

Este ejercicio no busca ser inteligente. Busca medir el punto de partida.

## Qué necesitás saber antes

Venís de ejercicios donde empezamos a separar responsabilidades. Antes de construir sistemas más complejos, necesitamos una medición base.

Conceptos:

- **Baseline:** versión simple contra la que comparamos mejoras futuras.
- **Benchmark:** lista fija de casos de prueba.
- **Accuracy:** cantidad de aciertos sobre el total.
- **Consulta mixta:** una consulta que pertenece a más de un dominio.

Por qué importa: si no medimos el bot simple, después no sabemos si el sistema multiagente realmente mejoró algo.

## Instalación e imports

Este notebook usa Python estándar, sin API key y sin LLM.

Eso es intencional: E10 es una medición de baseline. Queremos separar el problema de arquitectura antes de sumar modelos, LangGraph o RAG real.

In [ ]:
# Este ejercicio no usa API key ni LLM.
# La idea es medir un baseline antes de construir un sistema multiagente.

# TypedDict documenta la forma de los resultados.
# Literal limita los dominios esperados a etiquetas conocidas.
from typing import TypedDict, Literal

# json se deja disponible para mostrar que los resultados podrían serializarse o guardarse.
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")

## Sección 1 - Baseline empresarial

AcmeOps tiene soporte de HR, Tech y Billing.

En este baseline todos los documentos están mezclados y un único agente debe:

1. Leer la consulta.
2. Predecir un dominio.
3. Responder con documentos de ese dominio.

Esta arquitectura es simple, pero tiene un límite: cuando una consulta toca dos dominios, el agente único tiene que elegir o inventar una política de mezcla.

## Datos y benchmark

Creamos dos cosas:

- `mixed_docs`: documentos cortos mezclados por dominio.
- `benchmark_queries`: consultas con la etiqueta esperada.

Incluimos casos normales, un caso mixto y un caso fuera de alcance para medir el comportamiento real del baseline.

In [ ]:
# Documentos mezclados de tres áreas.
# El baseline tendrá que elegir un solo dominio y responder desde estos documentos.
mixed_docs = [
    ("hr", "Vacaciones: 15 días."),
    ("hr", "Seguro desde el primer día."),
    ("tech", "VPN: reiniciar cliente y validar MFA."),
    ("tech", "Contraseña: portal de identidad."),
    ("billing", "Facturas antes del día 25."),
    ("billing", "Reembolsos con recibo y centro de costo."),
]

# Benchmark fijo: cada tupla tiene (consulta, etiqueta esperada).
# Incluye casos simples, un caso mixto y un caso fuera de alcance.
benchmark_queries = [
    ("vacaciones", "hr"),
    ("seguro", "hr"),
    ("vpn", "tech"),
    ("contraseña", "tech"),
    ("factura", "billing"),
    ("reembolso", "billing"),
    ("vacaciones y vpn", "mixed"),
    ("almuerzo", "unknown"),
    ("recibo", "billing"),
    ("notebook", "tech"),
]

print("Docs:", len(mixed_docs))
print("Benchmark:", len(benchmark_queries), "consultas")

## Sección 2 - Agente único

En Resolution, `baseline_agent` usa reglas simples por palabras clave.

La parte importante es que detecta todos los dominios encontrados. Si encuentra más de uno, devuelve `mixed`.

Esto deja expuesto el problema que justificará ejercicios multiagente: una consulta mixta no debería depender de un único bloque de lógica.

In [ ]:
def baseline_agent(query: str) -> dict:
    # Este baseline usa reglas simples por palabras clave.
    # No es el objetivo final del curso: sirve para medir el límite de un agente único.
    text = query.lower()

    # Cada lista representa señales de un dominio.
    hr_words = ["vacaciones", "seguro"]
    tech_words = ["vpn", "contraseña", "notebook"]
    billing_words = ["factura", "reembolso", "recibo"]

    # Detectamos todos los dominios que aparecen en la consulta.
    matched = []
    if any(w in text for w in hr_words):
        matched.append("hr")
    if any(w in text for w in tech_words):
        matched.append("tech")
    if any(w in text for w in billing_words):
        matched.append("billing")

    # Si no hay match, el bot no sabe responder.
    if not matched:
        pred = "unknown"
        context = []

    # Si hay más de un dominio, marcamos el límite del baseline.
    # Un agente único no sabe delegar a varios especialistas todavía.
    elif len(matched) > 1:
        pred = "mixed"
        context = [doc for domain, doc in mixed_docs if domain in matched]

    # Caso simple: un solo dominio.
    else:
        pred = matched[0]
        context = [doc for domain, doc in mixed_docs if domain == pred]

    answer = " | ".join(context) or "No sé responder con confianza."
    return {"predicted_domain": pred, "answer": answer}

## Sección 3 - Resultados

Ejecutamos el benchmark y calculamos accuracy.

La salida se lee así:

```text
esperado | pred=predicho | ok=True/False | consulta
```

No buscamos una métrica perfecta. Buscamos una señal clara de dónde funciona y dónde no.

In [ ]:
# Ejecutamos el benchmark completo.
# Accuracy = cantidad de predicciones correctas / total de consultas.
results = []

for q, expected in benchmark_queries:
    r = baseline_agent(q)
    ok = r["predicted_domain"] == expected
    results.append(ok)

    # Imprimimos una fila por caso para que la clase vea dónde acierta y dónde falla.
    print(f"{expected:8} | pred={r['predicted_domain']:8} | ok={ok} | {q}")

print("Accuracy baseline:", sum(results), "/", len(results))

## Checks automáticos

Los checks validan el contrato mínimo:

- El benchmark tiene 10 casos.
- `baseline_agent` devuelve un diccionario.
- El diccionario contiene `predicted_domain` y `answer`.

En Starter deben pasar aunque el resultado sea malo, porque el objetivo es que el alumno pueda iterar sobre la función.

In [ ]:
def run_checks():
    # El benchmark debe tener 10 casos para comparar contra próximas versiones.
    assert len(benchmark_queries) == 10

    # El agente siempre debe devolver un dict con las claves esperadas.
    sample = baseline_agent("vpn")
    assert isinstance(sample, dict)
    assert "predicted_domain" in sample
    assert "answer" in sample

    print("Checks E10 OK")

run_checks()

## Qué aprendiste hoy

- Un baseline simple ayuda a medir mejoras futuras.
- Un agente único puede funcionar en casos simples.
- Las consultas mixtas exponen la necesidad de orquestación.
- Un contrato de salida estable facilita checks y comparación.

Frase para clase:

> Antes de construir un sistema multiagente, medimos el sistema simple. Si no sabemos dónde estamos parados, no podemos demostrar que mejoramos.